# TechMind — EDA del dataset ArXiv
Este notebook reproduce el análisis de `scripts/eda.py`: calidad, categorías, fechas, longitud de texto, coocurrencia y términos frecuentes.

In [ ]:
from pathlib import Path
import re
from itertools import combinations
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent
DATA_PATH = ROOT / 'data' / 'arxiv.csv'
REPORT_DIR = ROOT / 'reports' / 'eda'
RANDOM_STATE = 42
df = pd.read_csv(DATA_PATH)
df.info()
df.head()

In [ ]:
nulls = df.isna().sum().sort_values(ascending=False)
duplicates = {'paper_id': df['paper_id'].duplicated().sum(), 'title': df['title'].duplicated().sum(), 'abstract': df['abstract'].duplicated().sum(), 'rows': df.duplicated().sum()}
print('Nulos\n', nulls)
print('Duplicados\n', duplicates)
df['year_parsed'] = pd.to_datetime(df['year'], errors='coerce').dt.year
df['title_length'] = df['title'].fillna('').astype(str).str.len()
df['abstract_length'] = df['abstract'].fillna('').astype(str).str.len()
df[['title_length', 'abstract_length']].describe()

In [ ]:
sample = df.sample(n=min(20000, len(df)), random_state=RANDOM_STATE)
category_counts = df['primary_category'].value_counts()
display(category_counts.head(20))
plt.figure(figsize=(12, 7))
sns.barplot(y=category_counts.head(20).index, x=category_counts.head(20).values, color='#0d6efd')
plt.title('Top 20 categorías primarias')
plt.show()

In [ ]:
stopwords = {'the', 'and', 'for', 'with', 'from', 'that', 'this', 'using', 'their', 'which', 'based'}
tokens = re.findall(r'[a-zA-Z][a-zA-Z0-9-]{2,}', ' '.join(sample['abstract'].fillna('')).lower())
terms = pd.Series([token for token in tokens if token not in stopwords]).value_counts().head(30)
terms.sort_values().plot.barh(figsize=(10, 8), color='#fd7e14', title='Términos más frecuentes')
plt.show()

In [ ]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
category_counts.rename_axis('primary_category').reset_index(name='count').to_csv(REPORT_DIR / 'primary_categories.csv', index=False)
df[['title_length', 'abstract_length']].describe().to_csv(REPORT_DIR / 'text_length_stats.csv')
print(f'Reportes guardados en {REPORT_DIR}')